## TabFM Model – 1.0.0

### Description:

In this notebook I will build a classification and regression model using the TabFM foundation model. TabFM is a zero-shot tabular foundation model from Google Research. It supports classification and regression on structured data with mixed numerical and categorical columns without requiring fine-tuning or hyperparameter search, training examples are passed as context, and predictions are made in a single forward pass.


In [1]:
## This notebook is designed to be run on a local machine as well as Google Colab
# Base on the running platform, imports and file paths may change. To solve this, the running_local variable will be used to easily change the running platform logic.
running_local = True


## 1.0 Import  libraries

In [2]:
# utils folder is in the root, the notebook cannot unless the root path is added to the notebook.
############ import utils from if running from local machine ###########
if running_local:
    import sys
    sys.path.append('..')
    from utils.data_utils import get_model_metrics, predict_with_threshold, evaluate_regression_model
else:
    ###############################################################

    ################################################################
    ####################### GOOGLE COLAB RUN #######################
    ### Utils file is not available if running on Colab. I need to clone the repo to use the data_utils.py
    !git clone https://github.com/weslylaboy/student-outcome-predictor.git
    import sys
    sys.path.insert(0, '/content/student-outcome-predictor')
    from utils.data_utils import evaluate_regression_model
    ###############################################################

import pandas as pd

from sklearn.model_selection import train_test_split

from tabfm import TabFMClassifier, TabFMRegressor, tabfm_v1_0_0_pytorch as tabfm_v1_0_0
# import torch
# import joblib




### 1.1 Load data

The cleaned data was saved to `./data/processed/02.7_data_final.csv`. I will load this dataset for our modeling process.


In [4]:
if running_local:
    ##### import data file if running on local machine
    df = pd.read_csv('../data/processed/02.7_data_final.csv')
else:
    ##################### GOOGLE COLAB ################################
    ### Data location is different if running on colab
    df = pd.read_csv('/content/student-outcome-predictor/data/processed/02.7_data_final.csv')
    ###################################################################

df.head()

,marital_status,application_mode,application_order,course,daytime_evening,prev_qualification,prev_grade,nationality,mother_qualification,father_qualification,...,father_occupation_is_unknown,mother_occupation_is_unknown,father_qualification_is_unknown,mother_qualification_is_unknown,prev_qualification_is_unknown,any_info_missing,target_encoded,age_group,grade_progression,total_approved_units
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0,0,0,18-21,0.000000,0
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,0,0,0,0,0,2,18-21,-0.333333,12
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,0,0,0,0,0,0,18-21,0.000000,0
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,0,0,0,0,0,2,18-21,-1.028571,11
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,0,0,0,0,0,2,31+,0.666667,11


### 1.2 Fix Datatypes

In [5]:
# Fix int datatypes
int_cols = ['displaced', 'special_needs', 'debtor', 'tuition_fees_up_to_date', 'gender', 'scholarship_holder', 'international', 'daytime_evening', 'age_at_enrollment', 'cu1_credited', 'cu1_enrolled', 'cu1_evaluations', 'cu1_approved', 'cu1_no_evaluations', 'cu2_credited', 'cu2_enrolled', 'cu2_evaluations', 'cu2_approved', 'cu2_no_evaluations', 'total_approved_units', 'prev_qualification']
df[int_cols] = df[int_cols].astype('int8')

# additional engineered features
unknown_flags = [
    'father_occupation_is_unknown', 'mother_occupation_is_unknown',
    'father_qualification_is_unknown', 'mother_qualification_is_unknown',
    'prev_qualification_is_unknown',
    'any_info_missing'
]

df[unknown_flags] = df[unknown_flags].astype('int8')

# target variable
df['target_encoded'] = df['target_encoded'].astype('int8')


####### Fix binary data types
categorical_cols = [
    'marital_status',
    'application_mode',
    'application_order',
    'course',
    'nationality',
    'mother_qualification',
    'father_qualification',
    'mother_occupation',
    'father_occupation'
]
df[categorical_cols] = df[categorical_cols].astype('category')

# new engineered feature as category
df['age_group'] = df['age_group'].astype('category')

df.info()
df.head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 47 columns):
 #   Column                           Non-Null Count  Dtype   
---  ------                           --------------  -----   
 0   marital_status                   4424 non-null   category
 1   application_mode                 4424 non-null   category
 2   application_order                4424 non-null   category
 3   course                           4424 non-null   category
 4   daytime_evening                  4424 non-null   int8    
 5   prev_qualification               4424 non-null   int8    
 6   prev_grade                       4424 non-null   float64 
 7   nationality                      4424 non-null   category
 8   mother_qualification             4424 non-null   category
 9   father_qualification             4424 non-null   category
 10  mother_occupation                4424 non-null   category
 11  father_occupation                4424 non-null   category
 12  admiss

,marital_status,application_mode,application_order,course,daytime_evening,prev_qualification,prev_grade,nationality,mother_qualification,father_qualification,...,father_occupation_is_unknown,mother_occupation_is_unknown,father_qualification_is_unknown,mother_qualification_is_unknown,prev_qualification_is_unknown,any_info_missing,target_encoded,age_group,grade_progression,total_approved_units
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0,0,0,18-21,0.000000,0
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,0,0,0,0,0,2,18-21,-0.333333,12
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,0,0,0,0,0,0,18-21,0.000000,0
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,0,0,0,0,0,2,18-21,-1.028571,11
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,0,0,0,0,0,2,31+,0.666667,11


***Interpretation:*** I have fix the datatypes for all columns in dataset.

### 1.3 Data Preparation: Remove Target Columns

In this step I will separate the features from the target columns. Also, I will filter out `Enrolled` students as these students will be used to deploy the model. So, the model will be trained using students with status `Graduate` or `Dropout`.

In [6]:
# create a copy excluding students with status Enrolled
df_baseline = df[df['target_encoded'] != 1].copy()

# define features and target
X = df_baseline.drop(['target', 'target_encoded'], axis=1)

# update the encoded values to mapping it to 0 and 1
# 0 = Dropout, 1 = Graduate to have a clean binary column.
y = df_baseline['target_encoded'].map({0:0, 2:1})
X.shape, y.shape


((3630, 45), (3630,))

***Interpretation:*** The final dataset contains `3,630 rows`, `45 features`, and `1 target` variable.

In [7]:
X_train, X_test, y_train,y_test = train_test_split(X,y, test_size=0.2, stratify=y, random_state=42)

(X_train.shape, y_train.shape), (X_test.shape, y_test.shape)

(((2904, 45), (2904,)), ((726, 45), (726,)))

## 2.0 TabFM Classification Model

### 2.1 Pytorch Backend

In [46]:
model = tabfm_v1_0_0.load(checkpoint_path="../models/tabfm/classification/pytorch_model.bin", model_type="classification")

# The model object is a PyTorch Module, move it directly to MPS
# if torch.backends.mps.is_available():
#     model.to("mps")
#     print("Model moved to MPS (Mac GPU)")
# else:
#     print("MPS not available, using CPU")

clf = TabFMClassifier(model=model)
clf.fit(X_train, y_train)
clf


TabFMClassifier(model=TabFM(
  (cell_embedder): CellEmbedder(
    (in_linear): Linear(in_features=64, out_features=256, bias=True)
    (in_linear_cat): Linear(in_features=64, out_features=256, bias=True)
    (y_embedder_lookup): Embedding(10, 256)
  )
  (col_embedder): ColEmbedding(
    (tf_col): SetTransformer(
      (blocks): ModuleList(
        (0-2): 3 x InducedSelfAttentionBlock(
          (mab1): MultiheadAttenti...
          (linear1_gate): Linear(in_features=2048, out_features=8192, bias=True)
          (linear2): Linear(in_features=8192, out_features=2048, bias=True)
        )
      )
    )
    (ln): RMSNorm()
    (y_encoder): OneHotAndLinear(
      (projection): Linear(in_features=10, out_features=2048, bias=True)
    )
    (decoder): MLP(
      (layers): ModuleList(
        (0): Linear(in_features=2048, out_features=4096, bias=True)
        (1): Linear(in_features=4096, out_features=10, bias=True)
      )
    )
  )
))

In [47]:
custom_threshold = 0.4
y_pred_classification = predict_with_threshold(clf, X_test, y_test, threshold=custom_threshold, model_name='TabFM Classification Model')

KeyboardInterrupt: 

***Interpretation:*** The TabFM classification model performed really well, it have the best `recall` for `Dropout` class with `87%` compared to the `86%` recall for the XGBoost model. The TabFM model correctly identified `246 Dropout` and missclassifiying only `38 Dropouts` as `Graduate`, the XGBoost model correctly identified `245` and misclassified `38` `Dropout`.

The improvements are not really noticeable compared to the XGBoost model for this dataset. I would still choose the XGBoost model for this task as the XGBoost model predictions was faster than the TabFM. The TabFM model took `44 minutes` to generate predictions for `726 students`, the XGBoost model took less than a minute to generate the same predictions.

Since the TabFM model didn't show lots of improvements in performance and took more time to predict than the XGBoost model, I will keep using the XGBoost model to predict student outcome.

### 2.2 Saving Classification Model Result

In [ ]:
results_models_metrics = []

results_models_metrics.append(get_model_metrics(y_test, y_pred_classification, 'Tab FM Classification'))

temp_results = pd.DataFrame(results_models_metrics).set_index('Model')
temp_results

***Interpretation:***

### 3.0 TabFM Regression Model

The target variable for the regression model is `cu2_grade`.

In [8]:
target = 'cu2_grade'

### 3.1 Load data

To do a fair comparison of the regression model, I will use the classification model selected in the classification notebook. This model is saved in the `models` folders. I will import this model to identify the students at risk of `Dropout` and then use the TabFM regression model to predict the `GPA` for those students at risk.

In [14]:
if running_local:
    ##### import data file if running on local machine
    df = pd.read_csv('../data/validation/04_2.0_student_at_risk.csv')
else:
    ##################### GOOGLE COLAB ################################
    ### Data location is different if running on colab
    df = pd.read_csv('/content/student-outcome-predictor/data/validation/04_2.0_student_at_risk.csv')
    ###################################################################


print(f"Students flagged as dropout risk    : {len(df)}")
df.info()


Students flagged as dropout risk    : 1011
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1011 entries, 0 to 1010
Data columns (total 48 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   marital_status                   1011 non-null   int64  
 1   application_mode                 1011 non-null   int64  
 2   application_order                1011 non-null   int64  
 3   course                           1011 non-null   int64  
 4   daytime_evening                  1011 non-null   int64  
 5   prev_qualification               1011 non-null   int64  
 6   prev_grade                       1011 non-null   float64
 7   nationality                      1011 non-null   int64  
 8   mother_qualification             1011 non-null   int64  
 9   father_qualification             1011 non-null   int64  
 10  mother_occupation                1011 non-null   int64  
 11  father_occupation                1011 n

***Interpretation:*** I have loaded both the dataset and the classification model. The classification model was saved at the end of notebook 03 and I need it here to generate dropout probabilities for each student. The dataset has `47 columns` and the `target` for this notebook is `cu2_grade`, which is the average grade a student got in their second semester on a scale from `0 to 20`.


In [15]:
# Fix int datatypes
int_cols = ['displaced', 'special_needs', 'debtor', 'tuition_fees_up_to_date', 'gender', 'scholarship_holder', 'international', 'daytime_evening', 'age_at_enrollment', 'cu1_credited', 'cu1_enrolled', 'cu1_evaluations', 'cu1_approved', 'cu1_no_evaluations', 'cu2_credited', 'cu2_enrolled', 'cu2_evaluations', 'cu2_approved', 'cu2_no_evaluations', 'total_approved_units', 'prev_qualification']
df[int_cols] = df[int_cols].astype('int8')

# additional engineered features
unknown_flags = [
    'father_occupation_is_unknown', 'mother_occupation_is_unknown',
    'father_qualification_is_unknown', 'mother_qualification_is_unknown',
    'prev_qualification_is_unknown',
    'any_info_missing'
]

df[unknown_flags] = df[unknown_flags].astype('int8')

# target variable
df['target_encoded'] = df['target_encoded'].astype('int8')


####### Fix binary data types
categorical_cols = [
    'marital_status',
    'application_mode',
    'application_order',
    'course',
    'nationality',
    'mother_qualification',
    'father_qualification',
    'mother_occupation',
    'father_occupation'
]
df[categorical_cols] = df[categorical_cols].astype('category')

# new engineered feature as category
df['age_group'] = df['age_group'].astype('category')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1011 entries, 0 to 1010
Data columns (total 48 columns):
 #   Column                           Non-Null Count  Dtype   
---  ------                           --------------  -----   
 0   marital_status                   1011 non-null   category
 1   application_mode                 1011 non-null   category
 2   application_order                1011 non-null   category
 3   course                           1011 non-null   category
 4   daytime_evening                  1011 non-null   int8    
 5   prev_qualification               1011 non-null   int8    
 6   prev_grade                       1011 non-null   float64 
 7   nationality                      1011 non-null   category
 8   mother_qualification             1011 non-null   category
 9   father_qualification             1011 non-null   category
 10  mother_occupation                1011 non-null   category
 11  father_occupation                1011 non-null   category
 12  admiss

### 3.2 Train-Test split

In this step, I will split the student at risk data into training and test sets.


In [16]:
target = 'cu2_grade'

# Features available before the target is known
regression_features = [
    'marital_status',
    'application_mode',
    'application_order',
    'course',
    'daytime_evening',
    'prev_qualification',
    'prev_grade',
    'nationality',
    'mother_qualification',
    'father_qualification',
    'mother_occupation',
    'father_occupation',
    'admission_grade',
    'displaced',
    'special_needs',
    'debtor',
    'tuition_fees_up_to_date',
    'gender',
    'scholarship_holder',
    'age_at_enrollment',
    'international',
    'cu1_credited',
    'cu1_enrolled',
    'cu1_evaluations',
    'cu1_approved',
    'cu1_grade',
    'cu1_no_evaluations',
    'unemployment_rate',
    'inflation_rate',
    'gdp',
    'father_occupation_is_unknown',
    'mother_occupation_is_unknown',
    'father_qualification_is_unknown',
    'mother_qualification_is_unknown',
    'prev_qualification_is_unknown',
    'any_info_missing',
    'age_group',
    'cu2_grade'
]

# use only the feature for regression
df = df[regression_features]


X = df.drop(columns=['cu2_grade'])
y = df['cu2_grade']

X.shape, y.shape

((1011, 37), (1011,))

***Interpretation:*** After removing the feature engineering columns and selecting only the regression columns, I have a dataset of `36 features` and `1 target` variable.

In [17]:
X = df.drop(columns=['cu2_grade'])
y = df['cu2_grade']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set : {X_train.shape[0]} students")
print(f"Test set     : {X_test.shape[0]} students")


Training set : 808 students
Test set     : 203 students


***Interpretation*** The training set have `808` students, and the test set have `203` students.

### 3.4 Building the TabFM Regression Model

In [18]:
reg_model = tabfm_v1_0_0.load(model_type="regression", checkpoint_path='../models/tabfm/regression/pytorch_model.bin')

reg = TabFMRegressor(model=reg_model)

reg.fit(X_train, y_train)

TabFMRegressor(model=TabFM(
  (cell_embedder): CellEmbedder(
    (in_linear): Linear(in_features=64, out_features=256, bias=True)
    (in_linear_cat): Linear(in_features=64, out_features=256, bias=True)
    (y_embedder_lookup): MLP(
      (layers): ModuleList(
        (0): Linear(in_features=1, out_features=6, bias=True)
        (1): Linear(in_features=6, out_features=256, bias=True)
      )
    )
  )
  (col_embedder): ColE...
          (linear2): Linear(in_features=8192, out_features=2048, bias=True)
        )
      )
    )
    (ln): RMSNorm()
    (y_encoder): MLP(
      (layers): ModuleList(
        (0): Linear(in_features=1, out_features=4096, bias=True)
        (1): Linear(in_features=4096, out_features=2048, bias=True)
      )
    )
    (decoder): MLP(
      (layers): ModuleList(
        (0): Linear(in_features=2048, out_features=4096, bias=True)
        (1): Linear(in_features=4096, out_features=1, bias=True)
      )
    )
  )
))

### 3.5 Evaluating TabFM Regression Model

In [19]:
results = {}

y_pred_reg = evaluate_regression_model('TabFM Regression Model', reg, X_test, y_test, results)

pd.DataFrame(results)

[TabFM Regression Model] RMSE: 1.0635 | MAE: 0.8618 | R²: 0.3305


,TabFM Regression Model
RMSE,1.0635
MAE,0.8618
R²,0.3305


***Interpretation:*** The TabFM regressor model have a R2 of `0.3305` meaning that the model explain 33.05% of the GPA variance among students at risk, wich is an improvement o ver the 32.41% achieved by the Random Forest. This suggest that the foundation model architecture is slightly better at capturing the complex non-linear relationship within this dataset.

TabFM achieved a lower `RMSE` (`1.0635 vs 1.0686`). Since `RMSE` penalizes larger errors more heavily, this indicates that TabFM makes slightly fewer mispredictions compared to the Random Forest.

The MAE for TabFM is `0.8618`, meaning its typical prediction is off by approximately `0.86 GPA` points. This is slightly more precise than the Random Forest's `0.87` points.


## Analysis of Results and Conclusions

For the classification model, while the XGBoost model remains a very strong candidate due to its efficiency and high recall, the **TabFM Classification Model** achieved the highest recall for the `Dropout` class (`87%`). This makes it a powerful zero-shot alternative for early intervention systems where identifying every student at risk is the primary objective. However, given that the performance gains over traditional gradient-boosted trees are marginal and the computational cost is significantly higher, XGBoost remains the more practical choice for real-time deployment on this specific dataset.

For the regression model, while the Random Forest (Tuned) was previously identified as the best traditional machine learning model for this task, the **TabFM Regression Model** now stands as the overall top performer for predicting student GPA. Also, the performance for the TabFM regressor was good. The improvement is consistent across all metrics, confirming that the zero-shot foundation model approach provides a superior fit for this specific homogeneous subgroup of students flagged as being at risk of dropout.
